# Agente inteligente para gestion de prestamos y cobros

Caso E: Digitalizacion microfinanciera para bodegas y pequenos emprendimientos.

El cuaderno implementa un agente conversacional con LangChain para registrar prestamos/fiados, gestionar cobros, consultar clientes, calcular indicadores, generar graficos y revisar una base historica de movimientos.


## 1. Instalacion de dependencias

Esta seccion instala las librerias necesarias para ejecutar el agente en Google Colab.


In [ ]:
!pip uninstall -y -q langchain langchain-core langchain-google-genai langchain-openai langgraph langgraph-prebuilt google-generativeai google-ai-generativelanguage
!pip install -q "pandas==2.2.2" matplotlib seaborn ipywidgets
!pip install -q "langchain==0.3.27" "langchain-openai==0.3.28"

## 2. Configuracion del modelo LLM

El agente usa OpenRouter mediante LangChain. La API key se ingresa de forma oculta para no dejar credenciales escritas en el notebook. Las claves de OpenRouter suelen empezar con `sk-or-v1-`.

In [ ]:
import os
from getpass import getpass

os.environ.pop("GOOGLE_API_KEY", None)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Ingresa tu OPENROUTER_API_KEY: ").strip()

if not os.environ["OPENROUTER_API_KEY"].startswith("sk-or-v1-"):
    print("Advertencia: la clave no parece ser de OpenRouter. Normalmente empieza con sk-or-v1-")

# Compatibilidad: ChatOpenAI busca a veces OPENAI_API_KEY por defecto.
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]

print("API key cargada en memoria. No se mostrara por seguridad.")

## 3. Base historica y reglas de negocio

La logica administrativa se organiza en bloques pequenos: rutas de archivos, base de datos, reglas de cliente y seed de prueba. Esta organizacion facilita explicar el funcionamiento del agente durante la sustentacion.


### 3.1 Rutas, librerias y seguridad de sesion

Este bloque importa librerias, define la ubicacion de los archivos generados y configura un inicio de sesion simple para el bodeguero. La validacion de usuario simula una capa basica de seguridad antes de registrar o consultar datos sensibles.


In [ ]:

from datetime import datetime, date, timedelta
from pathlib import Path
from typing import Dict, List, Any, Optional
import sqlite3
import re
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = None

# Directorio donde quedaran la base historica, CSVs y graficos.
# En Colab se guarda en /content/agente_fiados_data; en local, dentro de la carpeta del notebook.
DATA_DIR = Path('/content/agente_fiados_data') if Path('/content').exists() else Path.cwd() / 'agente_fiados_data'
GRAPH_DIR = DATA_DIR / 'graficos'
DB_PATH = DATA_DIR / 'bodega_fiados_historico.sqlite'
CSV_CLIENTES = DATA_DIR / 'clientes_historico.csv'
CSV_PRESTAMOS = DATA_DIR / 'prestamos_historico.csv'
CSV_MOVIMIENTOS = DATA_DIR / 'movimientos_historico.csv'
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

LIMITES_RECOMENDADOS = {'AD': 300.0, 'A': 250.0, 'B': 180.0, 'C': 100.0, 'D': 0.0, 'NUEVO': 150.0}




### 3.2 Base SQLite, validaciones y exportacion

Este bloque crea las tablas historicas, valida datos importantes como DNI y telefono, y exporta la informacion a CSV para su revision externa.


In [ ]:
def conectar_db() -> sqlite3.Connection:
    """Abre una conexion SQLite hacia la base historica del prototipo."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def inicializar_base_historica() -> None:
    """Crea las tablas historicas si aun no existen."""
    with conectar_db() as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS clientes (
            dni TEXT PRIMARY KEY,
            nombre TEXT NOT NULL,
            telefono TEXT,
            limite_credito REAL NOT NULL DEFAULT 150,
            saldo REAL NOT NULL DEFAULT 0,
            estado_crediticio TEXT NOT NULL DEFAULT 'ACTIVO',
            categoria_pago TEXT NOT NULL DEFAULT 'NUEVO',
            fecha_alta TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS prestamos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            dni TEXT NOT NULL,
            cliente TEXT NOT NULL,
            monto REAL NOT NULL,
            concepto TEXT,
            fecha_prestamo TEXT NOT NULL,
            fecha_vencimiento TEXT NOT NULL,
            monto_pagado REAL NOT NULL DEFAULT 0,
            fecha_pago TEXT,
            estado TEXT NOT NULL DEFAULT 'PENDIENTE',
            FOREIGN KEY (dni) REFERENCES clientes(dni)
        );
        CREATE TABLE IF NOT EXISTS movimientos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            tipo TEXT NOT NULL,
            dni TEXT,
            cliente TEXT,
            monto REAL NOT NULL,
            concepto TEXT,
            fecha TEXT NOT NULL,
            prestamo_id INTEGER,
            FOREIGN KEY (prestamo_id) REFERENCES prestamos(id)
        );
        """)
        conn.commit()


def crear_estado_inicial() -> Dict[str, Any]:
    """Crea la estructura de sesion usada por las herramientas."""
    return {
        'clientes': {}, 'prestamos': [], 'movimientos': [],
        'caja': {'monto_inicial': 0.0, 'ventas_contado': 0.0, 'cobros_recibidos': 0.0, 'gastos': 0.0, 'fiados_otorgados': 0.0}
    }


estado: Dict[str, Any] = crear_estado_inicial()

# Usuarios de demostracion para proteger el uso del agente.
# En una implementacion real, estos usuarios vivirian en una tabla segura con claves cifradas.
USUARIOS_BODEGA = {
    'admin': '1234',
    'bodeguero': '2026',
}

sesion_bodeguero = {'activa': False, 'usuario': None}


def iniciar_sesion_bodeguero(usuario: str, clave: str) -> str:
    """Valida credenciales simples antes de permitir usar el agente."""
    usuario = str(usuario or '').strip().lower()
    clave = str(clave or '').strip()
    if USUARIOS_BODEGA.get(usuario) == clave:
        sesion_bodeguero['activa'] = True
        sesion_bodeguero['usuario'] = usuario
        return f'Inicio de sesion correcto. Bienvenido, {usuario}.'
    sesion_bodeguero['activa'] = False
    sesion_bodeguero['usuario'] = None
    return 'Credenciales incorrectas. Intenta nuevamente.'


def normalizar_cliente(nombre: str) -> str:
    """Normaliza nombres para busquedas flexibles."""
    return re.sub(r'\s+', ' ', str(nombre or '').strip().lower())


def validar_dni(dni: str = '') -> str:
    """Valida que el DNI tenga exactamente 8 digitos."""
    dni = str(dni or '').strip()
    if not re.fullmatch(r'\d{8}', dni):
        raise ValueError('El DNI debe tener exactamente 8 digitos numericos.')
    return dni


def validar_telefono(telefono: str = '') -> str:
    """Valida un telefono peruano simple de 9 digitos."""
    telefono = str(telefono or '').strip().replace(' ', '')
    if not telefono:
        return ''
    if not re.fullmatch(r'\d{9}', telefono):
        raise ValueError('El telefono debe tener 9 digitos numericos.')
    return telefono


def obtener_fecha(fecha_texto: str = 'hoy') -> str:
    """Convierte una fecha textual simple a ISO YYYY-MM-DD."""
    texto = str(fecha_texto or 'hoy').strip().lower()
    if texto in ['hoy', 'actual', 'ahora']:
        return date.today().isoformat()
    try:
        return datetime.fromisoformat(texto).date().isoformat()
    except ValueError:
        return date.today().isoformat()


def sumar_dias(fecha_iso: str, dias: int) -> str:
    """Suma dias a una fecha ISO."""
    return (datetime.fromisoformat(fecha_iso).date() + timedelta(days=int(dias))).isoformat()


def formatear_soles(monto: float) -> str:
    """Formatea montos en soles para respuestas del agente."""
    return f'S/ {float(monto):,.2f}'


def formatear_telefono(telefono: str = '') -> str:
    """Muestra telefonos como 944 555 666 para que sean mas legibles."""
    telefono = str(telefono or '').strip().replace(' ', '')
    if len(telefono) == 9 and telefono.isdigit():
        return f'{telefono[:3]} {telefono[3:6]} {telefono[6:]}'
    return telefono or 'no registrado'


def describir_categoria(categoria: str) -> str:
    """Explica el significado de cada categoria de pago."""
    descripciones = {
        'AD': 'Pagan antes',
        'A': 'Puntuales',
        'B': 'Regulares',
        'C': 'Tardios',
        'D': 'Riesgosos',
        'NUEVO': 'Sin historial suficiente',
    }
    categoria = str(categoria or 'NUEVO').upper()
    return descripciones.get(categoria, 'Sin descripcion')



def nombre_seguro_archivo(texto: str) -> str:
    """Convierte nombres de cliente en nombres seguros para carpetas/archivos."""
    texto = normalizar_cliente(texto).replace(' ', '_')
    texto = re.sub(r'[^a-z0-9_]+', '', texto)
    return texto or 'cliente'


def carpeta_graficos_cliente(nombre: str, dni: str) -> Path:
    """Devuelve la carpeta propia del cliente para guardar graficos."""
    carpeta = GRAPH_DIR / f"{nombre_seguro_archivo(nombre)}_{validar_dni(dni)}"
    carpeta.mkdir(parents=True, exist_ok=True)
    return carpeta


def registrar_movimiento_db(tipo: str, dni: str, cliente: str, monto: float, concepto: str, fecha: str, prestamo_id: Optional[int] = None) -> None:
    """Guarda un movimiento historico: prestamo, pago, venta, cierre o actualizacion."""
    with conectar_db() as conn:
        conn.execute('INSERT INTO movimientos(tipo, dni, cliente, monto, concepto, fecha, prestamo_id) VALUES (?, ?, ?, ?, ?, ?, ?)',
                     (tipo, dni, cliente, float(monto), concepto, obtener_fecha(fecha), prestamo_id))
        conn.commit()


def exportar_dataset_csv() -> None:
    """Exporta la base historica a CSV para que el grupo pueda revisarla fuera del notebook."""
    with conectar_db() as conn:
        pd.read_sql_query('SELECT * FROM clientes ORDER BY nombre', conn).to_csv(CSV_CLIENTES, index=False)
        pd.read_sql_query('SELECT * FROM prestamos ORDER BY fecha_prestamo, id', conn).to_csv(CSV_PRESTAMOS, index=False)
        pd.read_sql_query('SELECT * FROM movimientos ORDER BY fecha, id', conn).to_csv(CSV_MOVIMIENTOS, index=False)


def cargar_estado_desde_db() -> None:
    """Sincroniza el diccionario de sesion con la base SQLite."""
    global estado
    inicializar_base_historica()
    nuevo = crear_estado_inicial()
    with conectar_db() as conn:
        clientes = pd.read_sql_query('SELECT * FROM clientes', conn)
        prestamos = pd.read_sql_query('SELECT * FROM prestamos', conn)
        movimientos = pd.read_sql_query('SELECT * FROM movimientos', conn)
    for row in clientes.to_dict('records'):
        nuevo['clientes'][row['dni']] = row
    nuevo['prestamos'] = prestamos.to_dict('records')
    nuevo['movimientos'] = movimientos.to_dict('records')
    nuevo['caja']['fiados_otorgados'] = float(prestamos['monto'].sum()) if not prestamos.empty else 0.0
    nuevo['caja']['cobros_recibidos'] = float(movimientos.loc[movimientos['tipo'].eq('pago'), 'monto'].sum()) if not movimientos.empty else 0.0
    nuevo['caja']['ventas_contado'] = float(movimientos.loc[movimientos['tipo'].eq('venta_contado'), 'monto'].sum()) if not movimientos.empty else 0.0
    estado = nuevo




### 3.3 Busqueda, categoria y estado del cliente

Este bloque define la busqueda de clientes por DNI o nombre, la categoria de pago y el estado crediticio. La categoria resume el comportamiento historico de pago; el estado crediticio resume la situacion actual del cliente.


In [ ]:
def buscar_cliente_db(cliente: str = '', dni: str = '') -> Dict[str, Any]:
    """Busca un cliente por DNI o por nombre. Si hay nombres repetidos, exige DNI."""
    inicializar_base_historica()
    with conectar_db() as conn:
        if str(dni or '').strip():
            dni_valido = validar_dni(dni)
            row = conn.execute('SELECT * FROM clientes WHERE dni = ?', (dni_valido,)).fetchone()
            if row:
                return dict(row)
            raise KeyError('Cliente no encontrado por DNI.')
        nombre = normalizar_cliente(cliente)
        if not nombre:
            raise ValueError('Indica el nombre o DNI del cliente.')
        rows = conn.execute('SELECT * FROM clientes').fetchall()
        coincidencias = [dict(r) for r in rows if normalizar_cliente(r['nombre']) == nombre]
        if len(coincidencias) == 1:
            return coincidencias[0]
        if len(coincidencias) > 1:
            raise ValueError('Hay mas de un cliente con ese nombre. Consulta usando DNI.')
        raise KeyError('Cliente no encontrado.')


def actualizar_estado_prestamos() -> None:
    """Marca prestamos vencidos cuando superan la fecha de pago esperada."""
    hoy = date.today()
    with conectar_db() as conn:
        rows = conn.execute("SELECT * FROM prestamos WHERE estado IN ('PENDIENTE', 'PARCIAL')").fetchall()
        for r in rows:
            venc = datetime.fromisoformat(r['fecha_vencimiento']).date()
            if hoy > venc:
                conn.execute("UPDATE prestamos SET estado = 'VENCIDO' WHERE id = ?", (r['id'],))
        conn.commit()


def categorizar_cliente_por_pago(dni: str) -> str:
    """Clasifica al cliente por comportamiento historico de pago: AD, A, B, C, D o NUEVO."""
    dni = validar_dni(dni)
    hoy = date.today()
    with conectar_db() as conn:
        prestamos = conn.execute('SELECT * FROM prestamos WHERE dni = ?', (dni,)).fetchall()
    if not prestamos:
        return 'NUEVO'
    retrasos_pagados = []
    mora_activa_max = 0
    for p in prestamos:
        venc = datetime.fromisoformat(p['fecha_vencimiento']).date()
        pendiente = float(p['monto']) - float(p['monto_pagado'])
        if p['fecha_pago']:
            pago = datetime.fromisoformat(p['fecha_pago']).date()
            retrasos_pagados.append((pago - venc).days)
        elif pendiente > 0 and hoy > venc:
            mora_activa_max = max(mora_activa_max, (hoy - venc).days)
    if mora_activa_max > 30:
        return 'D'
    if mora_activa_max > 15:
        return 'C'
    if not retrasos_pagados:
        return 'NUEVO'
    promedio = sum(retrasos_pagados) / len(retrasos_pagados)
    if promedio <= -3:
        return 'AD'
    if promedio <= 0:
        return 'A'
    if promedio <= 7:
        return 'B'
    if promedio <= 15:
        return 'C'
    return 'D'


def clasificar_estado_crediticio(dni: str) -> str:
    """Determina si el cliente esta ACTIVO, OBSERVACION, CRITICO o BLOQUEADO."""
    dni = validar_dni(dni)
    actualizar_estado_prestamos()
    with conectar_db() as conn:
        c = conn.execute('SELECT * FROM clientes WHERE dni = ?', (dni,)).fetchone()
        if not c:
            return 'ACTIVO'
        limite = float(c['limite_credito'])
        prestamos = conn.execute('SELECT * FROM prestamos WHERE dni = ?', (dni,)).fetchall()
    saldo = sum(max(float(p['monto']) - float(p['monto_pagado']), 0) for p in prestamos)
    categoria = categorizar_cliente_por_pago(dni)
    vencidos = [p for p in prestamos if p['estado'] == 'VENCIDO']
    if categoria == 'D' or saldo >= limite or len(vencidos) >= 2:
        return 'BLOQUEADO'
    if saldo >= limite * 0.70 or categoria == 'C' or len(vencidos) == 1:
        return 'CRITICO'
    if saldo >= limite * 0.40 or categoria == 'B':
        return 'OBSERVACION'
    return 'ACTIVO'


def recalcular_cliente(dni: str) -> Dict[str, Any]:
    """Actualiza saldo, categoria y estado del cliente despues de cada movimiento."""
    dni = validar_dni(dni)
    actualizar_estado_prestamos()
    with conectar_db() as conn:
        prestamos = conn.execute('SELECT * FROM prestamos WHERE dni = ?', (dni,)).fetchall()
        saldo = sum(max(float(p['monto']) - float(p['monto_pagado']), 0) for p in prestamos)
        categoria = categorizar_cliente_por_pago(dni)
        estado_crediticio = clasificar_estado_crediticio(dni)
        conn.execute('UPDATE clientes SET saldo = ?, categoria_pago = ?, estado_crediticio = ? WHERE dni = ?',
                     (round(saldo, 2), categoria, estado_crediticio, dni))
        conn.commit()
        cliente = conn.execute('SELECT * FROM clientes WHERE dni = ?', (dni,)).fetchone()
    exportar_dataset_csv()
    cargar_estado_desde_db()
    return dict(cliente)




### 3.4 Registro historico y seed de demostracion

Este bloque guarda clientes, prestamos, pagos y movimientos. Tambien define `cargar_seed_demo()`, funcion que inicializa la base con datos suficientes para probar consultas, categorias, contactos, resumenes y graficos.


In [ ]:
def guardar_o_actualizar_cliente(nombre: str, dni: str, telefono: str = '', limite_credito: Optional[float] = None) -> Dict[str, Any]:
    """Crea o actualiza datos maestros del cliente en la base historica."""
    dni = validar_dni(dni)
    telefono = validar_telefono(telefono)
    nombre = str(nombre or 'Cliente sin nombre').strip().title()
    limite = float(limite_credito) if limite_credito is not None else LIMITES_RECOMENDADOS['NUEVO']
    with conectar_db() as conn:
        existe = conn.execute('SELECT * FROM clientes WHERE dni = ?', (dni,)).fetchone()
        if existe:
            limite_final = float(limite_credito) if limite_credito is not None else float(existe['limite_credito'])
            telefono_final = telefono or existe['telefono'] or ''
            nombre_final = nombre if nombre != 'Cliente Sin Nombre' else existe['nombre']
            conn.execute('UPDATE clientes SET nombre = ?, telefono = ?, limite_credito = ? WHERE dni = ?',
                         (nombre_final, telefono_final, limite_final, dni))
        else:
            conn.execute('INSERT INTO clientes(dni, nombre, telefono, limite_credito, saldo, estado_crediticio, categoria_pago, fecha_alta) VALUES (?, ?, ?, ?, 0, ?, ?, ?)',
                         (dni, nombre, telefono, limite, 'ACTIVO', 'NUEVO', date.today().isoformat()))
        conn.commit()
    return recalcular_cliente(dni)


def insertar_prestamo_historico(dni: str, cliente: str, monto: float, concepto: str, fecha_prestamo: str, fecha_vencimiento: str, monto_pagado: float = 0.0, fecha_pago: str = '') -> int:
    """Inserta un prestamo historico. Se usa para seed y para la herramienta principal."""
    estado_prestamo = 'PAGADO' if float(monto_pagado) >= float(monto) else 'PARCIAL' if float(monto_pagado) > 0 else 'PENDIENTE'
    if estado_prestamo != 'PAGADO' and datetime.fromisoformat(fecha_vencimiento).date() < date.today():
        estado_prestamo = 'VENCIDO'
    sql = 'INSERT INTO prestamos(dni, cliente, monto, concepto, fecha_prestamo, fecha_vencimiento, monto_pagado, fecha_pago, estado) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)'
    with conectar_db() as conn:
        cur = conn.execute(sql, (dni, cliente, float(monto), concepto, fecha_prestamo, fecha_vencimiento, float(monto_pagado), fecha_pago or None, estado_prestamo))
        prestamo_id = int(cur.lastrowid)
        conn.commit()
    registrar_movimiento_db('prestamo', dni, cliente, monto, concepto, fecha_prestamo, prestamo_id)
    if float(monto_pagado) > 0:
        registrar_movimiento_db('pago', dni, cliente, monto_pagado, f'Pago historico de {concepto}', fecha_pago or fecha_prestamo, prestamo_id)
    recalcular_cliente(dni)
    return prestamo_id


def reiniciar_base_historica() -> str:
    """Limpia las tablas historicas sin borrar el archivo SQLite. Es mas estable en Windows y Colab."""
    inicializar_base_historica()
    with conectar_db() as conn:
        conn.execute('DELETE FROM movimientos')
        conn.execute('DELETE FROM prestamos')
        conn.execute('DELETE FROM clientes')
        conn.execute("DELETE FROM sqlite_sequence WHERE name IN ('movimientos', 'prestamos')")
        conn.commit()
    cargar_estado_desde_db()
    exportar_dataset_csv()
    return 'Base historica reiniciada.'


def cargar_seed_demo() -> str:
    """Carga un dataset amplio para probar consultas, categorias y graficos durante la exposicion."""
    reiniciar_base_historica()
    clientes = [
        ('12345678', 'Juan Perez', '987654321', 160), ('87654321', 'Carmen Ruiz', '912345678', 220),
        ('45678912', 'Luis Torres', '999888777', 120), ('11223344', 'Maria Lopez', '955666777', 260),
        ('99887766', 'Pedro Ramos', '944555666', 90), ('33445566', 'Ana Castillo', '933222111', 300),
        ('77889900', 'Carlos Vega', '922333444', 180), ('66778899', 'Rosa Flores', '911222333', 140),
    ]
    for dni, nombre, telefono, limite in clientes:
        guardar_o_actualizar_cliente(nombre, dni, telefono, limite)
    hoy = date.today()
    datos_prestamos = [
        ('12345678', 45, 'Aceite', -50, -20, 45, -25), ('12345678', 70, 'Arroz', -12, 18, 0, None),
        ('87654321', 120, 'Arroz y azucar', -40, -10, 80, -2), ('87654321', 60, 'Leche', -8, 22, 0, None),
        ('45678912', 80, 'Gaseosas', -70, -40, 30, -5), ('45678912', 50, 'Fideos', -35, -5, 0, None),
        ('11223344', 90, 'Abarrotes', -45, -15, 90, -18), ('11223344', 55, 'Bebidas', -20, 10, 55, 8),
        ('99887766', 85, 'Productos varios', -60, -30, 10, -20), ('99887766', 50, 'Panaderia', -25, -2, 0, None),
        ('33445566', 140, 'Canasta familiar', -35, -5, 140, -12), ('33445566', 65, 'Limpieza', -5, 25, 0, None),
        ('77889900', 75, 'Bebidas', -25, 5, 75, 5), ('66778899', 110, 'Abarrotes', -50, -20, 40, -1),
    ]
    with conectar_db() as conn:
        nombres = {r['dni']: r['nombre'] for r in conn.execute('SELECT dni, nombre FROM clientes')}
    for dni, monto, concepto, dias_inicio, dias_venc, monto_pagado, dias_pago in datos_prestamos:
        fecha_prestamo = (hoy + timedelta(days=dias_inicio)).isoformat()
        fecha_vencimiento = (hoy + timedelta(days=dias_venc)).isoformat()
        fecha_pago = (hoy + timedelta(days=dias_pago)).isoformat() if dias_pago is not None else ''
        insertar_prestamo_historico(dni, nombres[dni], monto, concepto, fecha_prestamo, fecha_vencimiento, monto_pagado, fecha_pago)
    exportar_dataset_csv()
    cargar_estado_desde_db()
    return f'Seed demo cargado: {len(clientes)} clientes, {len(datos_prestamos)} prestamos y {len(estado["movimientos"])} movimientos historicos.'


inicializar_base_historica()
cargar_estado_desde_db()
print(f'Base historica lista: {DB_PATH}')
print(f'CSVs exportables: {CSV_CLIENTES}, {CSV_PRESTAMOS}, {CSV_MOVIMIENTOS}')


## 4. Herramientas del agente

Cada herramienta representa una operacion del servicio: registrar clientes, registrar prestamos, registrar pagos, consultar deuda, categorizar clientes, generar reportes o crear graficos. Las docstrings indican al LLM cuando debe usar cada herramienta y que regla de negocio aplica.


In [ ]:

try:
    from langchain_core.tools import tool
except ImportError:
    from langchain.tools import tool


@tool
def registrar_cliente(cliente: str, dni: str, telefono: str = '', limite_credito: float = 150.0) -> str:
    """
    Registra o actualiza un cliente usando DNI de 8 digitos.
    Usala cuando el bodeguero quiera guardar nombre, DNI, telefono o limite de credito del cliente.
    """
    try:
        datos = guardar_o_actualizar_cliente(cliente, dni, telefono, limite_credito)
        return (
            f"Cliente registrado/actualizado: {datos['nombre']} | DNI: {datos['dni']} | "
            f"Telefono: {formatear_telefono(datos.get('telefono'))} | Limite: {formatear_soles(datos['limite_credito'])} | "
            f"Categoria: {datos['categoria_pago']} | Estado: {datos['estado_crediticio']}"
        )
    except Exception as e:
        return f"No se pudo registrar el cliente. Detalle: {e}"


@tool
def registrar_prestamo_fiado(cliente: str, dni: str, monto_prestamo: float, concepto: str = 'Sin detalle', telefono: str = '', fecha_prestamo: str = 'hoy', dias_plazo: int = 30) -> str:
    """
    Registra un prestamo/fiado solicitado por un cliente.
    Usala cuando el usuario diga que un cliente quiere llevar productos fiados, debe dinero o solicita un prestamo de bodega.
    Regla: si el nuevo saldo supera el limite de credito definido por el bodeguero, se rechaza el prestamo.
    """
    try:
        monto_prestamo = float(monto_prestamo)
        if monto_prestamo <= 0:
            return 'No se registro el prestamo: el monto debe ser mayor que cero.'
        datos = guardar_o_actualizar_cliente(cliente, dni, telefono or '', None)
        saldo_actual = float(datos['saldo'])
        limite = float(datos['limite_credito'])
        nuevo_saldo = saldo_actual + monto_prestamo
        if datos['estado_crediticio'] == 'BLOQUEADO' or nuevo_saldo > limite:
            contacto = datos.get('telefono') or 'telefono no registrado'
            return (
                f"Prestamo rechazado para {datos['nombre']}. Saldo actual: {formatear_soles(saldo_actual)}; "
                f"monto solicitado: {formatear_soles(monto_prestamo)}; nuevo saldo: {formatear_soles(nuevo_saldo)}; "
                f"limite permitido: {formatear_soles(limite)}. Estado: BLOQUEADO. Contacto: {contacto}."
            )
        fecha_ini = obtener_fecha(fecha_prestamo)
        fecha_venc = sumar_dias(fecha_ini, dias_plazo)
        insertar_prestamo_historico(datos['dni'], datos['nombre'], monto_prestamo, concepto, fecha_ini, fecha_venc)
        datos = recalcular_cliente(datos['dni'])
        alerta = ''
        if datos['estado_crediticio'] in ['CRITICO', 'BLOQUEADO']:
            alerta = f" Alerta: revisar cobranza. Contacto: {datos.get('telefono') or 'telefono no registrado'}."
        return (
            f"Prestamo/fiado registrado para {datos['nombre']} | DNI: {datos['dni']} | "
            f"Monto: {formatear_soles(monto_prestamo)} | Vence: {fecha_venc} | "
            f"Saldo actual: {formatear_soles(datos['saldo'])} | Categoria: {datos['categoria_pago']} | Estado: {datos['estado_crediticio']}.{alerta}"
        )
    except Exception as e:
        return f"No se pudo registrar el prestamo/fiado. Detalle: {e}"


@tool
def registrar_pago(cliente: str = '', dni: str = '', monto: float = 0.0, fecha_pago: str = 'hoy') -> str:
    """
    Registra un pago o abono contra los prestamos pendientes del cliente.
    Usala cuando el usuario diga que un cliente pago, abono o cancelo una deuda.
    El pago se aplica primero al prestamo mas antiguo pendiente.
    """
    try:
        monto = float(monto)
        if monto <= 0:
            return 'No se registro el pago: el monto debe ser mayor que cero.'
        datos = buscar_cliente_db(cliente, dni)
        restante = monto
        fecha = obtener_fecha(fecha_pago)
        aplicados = []
        with conectar_db() as conn:
            prestamos = conn.execute("SELECT * FROM prestamos WHERE dni = ? AND estado IN ('PENDIENTE', 'PARCIAL', 'VENCIDO') ORDER BY fecha_vencimiento, id", (datos['dni'],)).fetchall()
            if not prestamos:
                return f"{datos['nombre']} no tiene prestamos pendientes."
            for p in prestamos:
                if restante <= 0:
                    break
                pendiente = float(p['monto']) - float(p['monto_pagado'])
                pago_aplicado = min(restante, pendiente)
                nuevo_pagado = float(p['monto_pagado']) + pago_aplicado
                nuevo_estado = 'PAGADO' if nuevo_pagado >= float(p['monto']) else 'PARCIAL'
                fecha_cierre = fecha if nuevo_estado == 'PAGADO' else None
                conn.execute('UPDATE prestamos SET monto_pagado = ?, estado = ?, fecha_pago = COALESCE(?, fecha_pago) WHERE id = ?',
                             (round(nuevo_pagado, 2), nuevo_estado, fecha_cierre, p['id']))
                conn.execute('INSERT INTO movimientos(tipo, dni, cliente, monto, concepto, fecha, prestamo_id) VALUES (?, ?, ?, ?, ?, ?, ?)',
                             ('pago', datos['dni'], datos['nombre'], pago_aplicado, f"Pago aplicado a {p['concepto']}", fecha, p['id']))
                aplicados.append(f"{formatear_soles(pago_aplicado)} a prestamo #{p['id']} ({p['concepto']})")
                restante -= pago_aplicado
            conn.commit()
        datos = recalcular_cliente(datos['dni'])
        extra = f" Quedo un excedente de {formatear_soles(restante)}." if restante > 0 else ''
        return (
            f"Pago registrado para {datos['nombre']}. Aplicacion: {', '.join(aplicados)}. "
            f"Saldo actual: {formatear_soles(datos['saldo'])} | Categoria: {datos['categoria_pago']} | Estado: {datos['estado_crediticio']}.{extra}"
        )
    except Exception as e:
        return f"No se pudo registrar el pago. Detalle: {e}"


@tool
def consultar_deuda(cliente: str = '', dni: str = '') -> str:
    """
    Consulta deuda, prestamos pendientes, categoria de pago y estado crediticio de un cliente.
    Usala cuando el usuario pregunte cuanto debe un cliente o quiera ver su deuda actual.
    """
    try:
        datos = buscar_cliente_db(cliente, dni)
        datos = recalcular_cliente(datos['dni'])
        with conectar_db() as conn:
            prestamos = conn.execute("SELECT * FROM prestamos WHERE dni = ? ORDER BY fecha_vencimiento, id", (datos['dni'],)).fetchall()
        lineas = []
        for p in prestamos[-6:]:
            pendiente = max(float(p['monto']) - float(p['monto_pagado']), 0)
            lineas.append(f"- #{p['id']} | {p['concepto']} | monto {formatear_soles(p['monto'])} | pendiente {formatear_soles(pendiente)} | vence {p['fecha_vencimiento']} | estado {p['estado']}")
        detalle = '\n'.join(lineas) if lineas else 'Sin prestamos registrados.'
        return (
            f"Cliente: {datos['nombre']} | DNI: {datos['dni']} | Telefono: {formatear_telefono(datos.get('telefono'))}\n"
            f"Saldo pendiente: {formatear_soles(datos['saldo'])} | Categoria: {datos['categoria_pago']} | Estado: {datos['estado_crediticio']}\n"
            f"Prestamos recientes:\n{detalle}"
        )
    except Exception as e:
        return f"No se pudo consultar la deuda. Detalle: {e}"


@tool
def consultar_historial_cliente(cliente: str = '', dni: str = '') -> str:
    """
    Muestra movimientos historicos del cliente: prestamos, pagos y fechas.
    Usala cuando el usuario pida historial, tracking o comportamiento de un cliente.
    """
    try:
        datos = buscar_cliente_db(cliente, dni)
        with conectar_db() as conn:
            movs = conn.execute('SELECT * FROM movimientos WHERE dni = ? ORDER BY fecha DESC, id DESC LIMIT 10', (datos['dni'],)).fetchall()
        if not movs:
            return f"{datos['nombre']} no tiene movimientos historicos."
        detalle = '\n'.join(f"- {m['fecha']} | {m['tipo']} | {formatear_soles(m['monto'])} | {m['concepto']}" for m in movs)
        return f"Historial de {datos['nombre']} | DNI: {datos['dni']}\n{detalle}"
    except Exception as e:
        return f"No se pudo consultar el historial. Detalle: {e}"


@tool
def categorizar_cliente(cliente: str = '', dni: str = '') -> str:
    """
    Consulta la categoria de pago y el estado crediticio de un cliente.
    Devuelve una ficha breve para revisar riesgo, saldo y limite actual.
    """
    try:
        datos = buscar_cliente_db(cliente, dni)
        datos = recalcular_cliente(datos['dni'])
        return (
            f"Categoria del cliente:\n"
            f"- Cliente: {datos['nombre']}\n"
            f"- DNI: {datos['dni']}\n"
            f"- Categoria: {datos['categoria_pago']}\n"
            f"- Estado: {datos['estado_crediticio']}\n"
            f"- Saldo pendiente: {formatear_soles(datos['saldo'])}\n"
            f"- Limite actual: {formatear_soles(datos['limite_credito'])}"
        )
    except Exception as e:
        return f"No se pudo categorizar al cliente. Detalle: {e}"


@tool
def listar_principales_deudores(limite: int = 8) -> str:
    """
    Lista clientes ordenados por mayor saldo pendiente.
    Usala para ranking de deudores, morosos principales o clientes con mayor riesgo.
    """
    try:
        cargar_estado_desde_db()
        todos_deudores = [c for c in estado['clientes'].values() if float(c['saldo']) > 0]
        clientes = sorted(todos_deudores, key=lambda c: float(c['saldo']), reverse=True)[:max(1, int(limite))]
        if not clientes:
            return 'No hay clientes con deuda activa.'
        total_deuda = sum(float(c['saldo']) for c in todos_deudores)
        lineas = ["Principales deudores:"]
        for i, c in enumerate(clientes, 1):
            pct = float(c['saldo']) / total_deuda * 100 if total_deuda else 0
            lineas.append(f"{i}. {c['nombre']}")
            lineas.append(f"   - DNI: {c['dni']}")
            lineas.append(f"   - Deuda: {formatear_soles(c['saldo'])}")
            lineas.append(f"   - Porcentaje de la deuda pendiente total: {pct:.1f}%")
            lineas.append(f"   - Categoria: {c['categoria_pago']} ({describir_categoria(c['categoria_pago'])})")
            lineas.append(f"   - Estado: {c['estado_crediticio']}")
            if c.get('estado_crediticio') in ['CRITICO', 'BLOQUEADO'] and c.get('telefono'):
                lineas.append(f"   - Telefono: {formatear_telefono(c.get('telefono'))}")
        lineas.append("")
        return "\n".join(lineas)
    except Exception as e:
        return f"No se pudo listar deudores. Detalle: {e}"


@tool
def resumen_financiero() -> str:
    """
    Muestra un resumen financiero general con deuda, caja, recuperacion y estado de clientes.
    Usala cuando el bodeguero pregunte como va la bodega, cuanto dinero esta fuera de caja o pida resumen financiero.
    """
    try:
        cargar_estado_desde_db()
        clientes = list(estado['clientes'].values())
        deuda_total = sum(float(c['saldo']) for c in clientes)
        activos = sum(1 for c in clientes if float(c['saldo']) > 0)
        criticos = sum(1 for c in clientes if c['estado_crediticio'] in ['CRITICO', 'BLOQUEADO'])
        fiados = estado['caja']['fiados_otorgados']
        cobros = estado['caja']['cobros_recibidos']
        tasa = (cobros / fiados * 100) if fiados else 0
        caja_esperada = estado['caja']['monto_inicial'] + estado['caja']['ventas_contado'] + cobros - estado['caja']['gastos']
        return (
            "Resumen financiero general:\n"
            f"- Dinero fuera de caja: {formatear_soles(deuda_total)}\n"
            f"- Caja esperada (dinero que deberia haber en caja): {formatear_soles(caja_esperada)}\n"
            f"- Monto del historial de prestamos: {formatear_soles(fiados)}\n"
            f"- Monto del historial de cobros: {formatear_soles(cobros)}\n"
            f"- Tasa de recuperacion: {tasa:.1f}%\n"
            f"- Clientes con deuda activa: {activos}\n"
            f"- Clientes criticos o bloqueados: {criticos}\n\n"
            "Para ver la distribucion por categoria, pide: 'resumen de categorias'."
        )
    except Exception as e:
        return f"No se pudo generar resumen financiero. Detalle: {e}"


@tool
def resumen_categorias_clientes() -> str:
    """
    Resume cuantos clientes hay por categoria de pago: AD, A, B, C, D o NUEVO.
    Usala cuando el usuario pregunte por categorias, clientes puntuales, tardios, riesgosos o distribucion de clientes.
    """
    try:
        cargar_estado_desde_db()
        clientes = list(estado['clientes'].values())
        orden = ['AD', 'A', 'B', 'C', 'D', 'NUEVO']
        conteos = {cat: 0 for cat in orden}
        for c in clientes:
            cat = c.get('categoria_pago', 'NUEVO')
            conteos[cat] = conteos.get(cat, 0) + 1
        lineas = ["Resumen por categoria de pago:"]
        for cat in orden:
            n = conteos.get(cat, 0)
            lineas.append(f"- Categoria {cat} ({describir_categoria(cat)}): {n} clientes")
        lineas.append("")
        lineas.append("Lectura rapida:")
        lineas.append("- AD: clientes que pagan antes del vencimiento.")
        lineas.append("- A: clientes puntuales.")
        lineas.append("- B: clientes regulares con retrasos leves.")
        lineas.append("- C: clientes tardios.")
        lineas.append("- D: clientes riesgosos.")
        lineas.append("")
        lineas.append("Para ver nombres y DNI, pide: lista clientes de categoria A o clientes categoria D.")
        return "\n".join(lineas)
    except Exception as e:
        return f"No se pudo generar resumen por categorias. Detalle: {e}"


@tool
def listar_clientes_por_categoria(categoria: str) -> str:
    """
    Lista los clientes que pertenecen a una categoria de pago especifica: AD, A, B, C, D o NUEVO.
    Usala cuando el usuario pida consultar clientes de una categoria, por ejemplo clientes categoria A o clientes riesgosos.
    """
    try:
        cargar_estado_desde_db()
        categoria = str(categoria or '').strip().upper()
        alias = {
            'PUNTUALES': 'A', 'PUNTUAL': 'A',
            'ADELANTADOS': 'AD', 'ADELANTADO': 'AD',
            'REGULARES': 'B', 'REGULAR': 'B',
            'TARDIOS': 'C', 'TARDIO': 'C',
            'RIESGOSOS': 'D', 'RIESGOSO': 'D',
            'NUEVOS': 'NUEVO',
        }
        categoria = alias.get(categoria, categoria)
        validas = {'AD', 'A', 'B', 'C', 'D', 'NUEVO'}
        if categoria not in validas:
            return "Categoria no valida. Usa AD, A, B, C, D o NUEVO."
        clientes = [c for c in estado['clientes'].values() if c.get('categoria_pago') == categoria]
        if not clientes:
            return f"No hay clientes en categoria {categoria}."
        clientes = sorted(clientes, key=lambda c: float(c['saldo']), reverse=True)
        lineas = [f"Clientes en categoria {categoria}:"]
        for c in clientes:
            lineas.append(
                f"- {c['nombre']} | DNI: {c['dni']} | Deuda: {formatear_soles(c['saldo'])} | Estado: {c['estado_crediticio']} | Tel: {formatear_telefono(c.get('telefono'))}"
            )
        return "\n".join(lineas)
    except Exception as e:
        return f"No se pudo listar clientes por categoria. Detalle: {e}"


@tool
def consultar_contacto_cobranza(cliente: str = '', dni: str = '') -> str:
    """
    Devuelve telefono de contacto cuando un cliente esta CRITICO o BLOQUEADO.
    Usala cuando el bodeguero quiera llamar o cobrar a un cliente con deuda alta.
    """
    try:
        datos = buscar_cliente_db(cliente, dni)
        datos = recalcular_cliente(datos['dni'])
        if datos['estado_crediticio'] not in ['CRITICO', 'BLOQUEADO']:
            return (
                f"Contacto de cobranza para {datos['nombre']}:\n"
                f"- DNI: {datos['dni']}\n"
                f"- Estado actual: {datos['estado_crediticio']}\n"
                f"- Categoria: {datos['categoria_pago']} ({describir_categoria(datos['categoria_pago'])})\n"
                "- Accion sugerida: no requiere cobranza urgente."
            )
        return (
            f"Contacto de cobranza para {datos['nombre']}:\n"
            f"- DNI: {datos['dni']}\n"
            f"- Telefono: {formatear_telefono(datos.get('telefono'))}\n"
            f"- Deuda: {formatear_soles(datos['saldo'])}\n"
            f"- Categoria: {datos['categoria_pago']} ({describir_categoria(datos['categoria_pago'])})\n"
            f"- Estado: {datos['estado_crediticio']}\n"
            "- Accion sugerida: contactar al cliente y solicitar abono."
        )
    except Exception as e:
        return f"No se pudo consultar contacto. Detalle: {e}"


@tool
def configurar_limite_credito(cliente: str = '', dni: str = '', limite_credito: float = 150.0) -> str:
    """
    Cambia el limite de credito definido por el bodeguero para un cliente.
    Usala cuando el administrador quiera subir o bajar el monto maximo permitido para fiar.
    """
    try:
        datos = buscar_cliente_db(cliente, dni)
        datos = guardar_o_actualizar_cliente(datos['nombre'], datos['dni'], datos.get('telefono') or '', float(limite_credito))
        return f"Limite actualizado para {datos['nombre']}: {formatear_soles(datos['limite_credito'])}. Estado: {datos['estado_crediticio']} | Categoria: {datos['categoria_pago']}."
    except Exception as e:
        return f"No se pudo configurar el limite. Detalle: {e}"


@tool
def registrar_venta_contado(monto: float, concepto: str = 'Venta al contado', fecha: str = 'hoy') -> str:
    """
    Registra una venta pagada al momento para calcular caja esperada.
    Usala cuando se indique venta en efectivo, yapeada o cobrada al contado.
    """
    try:
        monto = float(monto)
        if monto <= 0:
            return 'No se registro la venta: el monto debe ser mayor que cero.'
        registrar_movimiento_db('venta_contado', '', '', monto, concepto, fecha)
        cargar_estado_desde_db()
        exportar_dataset_csv()
        return f"Venta al contado registrada por {formatear_soles(monto)}. Concepto: {concepto}."
    except Exception as e:
        return f"No se pudo registrar la venta al contado. Detalle: {e}"


@tool
def cerrar_caja(monto_inicial: float = 0.0, gastos: float = 0.0) -> str:
    """
    Calcula caja esperada usando monto inicial, ventas al contado, pagos cobrados y gastos.
    Usala para arqueo o cierre de turno.
    """
    try:
        estado['caja']['monto_inicial'] = float(monto_inicial)
        estado['caja']['gastos'] = float(gastos)
        caja = estado['caja']['monto_inicial'] + estado['caja']['ventas_contado'] + estado['caja']['cobros_recibidos'] - estado['caja']['gastos']
        registrar_movimiento_db('cierre_caja', '', '', caja, f'Cierre con gastos {gastos}', 'hoy')
        return f"Caja esperada (dinero que deberia haber en caja): {formatear_soles(caja)} | Monto inicial: {formatear_soles(monto_inicial)} | Gastos: {formatear_soles(gastos)}."
    except Exception as e:
        return f"No se pudo cerrar caja. Detalle: {e}"


@tool
def generar_grafico_morosidad(top_n: int = 8) -> str:
    """
    Genera un grafico de barras con clientes de mayor deuda.
    Usala cuando el usuario pida grafico, visualizacion o reporte visual de morosidad.
    """
    try:
        cargar_estado_desde_db()
        clientes = [c for c in estado['clientes'].values() if float(c['saldo']) > 0]
        if not clientes:
            return 'No hay deuda activa para graficar.'
        clientes = sorted(clientes, key=lambda c: float(c['saldo']), reverse=True)[:max(1, int(top_n))]
        df = pd.DataFrame({
            'Cliente': [f"{c['nombre']} (\"{c['categoria_pago']}\"/\"{c['estado_crediticio']}\")" for c in clientes],
            'Deuda': [float(c['saldo']) for c in clientes],
        })
        plt.figure(figsize=(8.5, 4.2))
        ax = sns.barplot(data=df, x='Deuda', y='Cliente', hue='Cliente', palette='viridis', legend=False)
        ax.set_title('Clientes con mayor deuda pendiente')
        ax.set_xlabel('Deuda pendiente (S/)')
        ax.set_ylabel('Cliente')
        max_deuda = float(df['Deuda'].max())
        ax.set_xlim(0, max_deuda * 1.18)
        for container in ax.containers:
            ax.bar_label(container, fmt='S/ %.2f', padding=3, fontsize=9)
        plt.tight_layout()
        
        fecha_archivo = date.today().isoformat()
        carpeta_general = GRAPH_DIR / 'reportes_generales'
        carpeta_general.mkdir(parents=True, exist_ok=True)
        ruta = carpeta_general / f'grafico_morosidad_{fecha_archivo}.png'
        plt.savefig(ruta, dpi=160, bbox_inches='tight')
        plt.close()
        return f"Grafico de morosidad generado. Ruta: {ruta}"
    except Exception as e:
        return f"No se pudo generar el grafico de morosidad. Detalle: {e}"


@tool
def generar_grafico_cliente(cliente: str = '', dni: str = '') -> str:
    """
    Genera un grafico historico de prestamos y pagos de un cliente especifico.
    Usala cuando el usuario pida la grafica o seguimiento visual de un cliente.
    """
    try:
        datos = buscar_cliente_db(cliente, dni)
        with conectar_db() as conn:
            movs = pd.read_sql_query('SELECT fecha, tipo, monto, concepto FROM movimientos WHERE dni = ? ORDER BY fecha, id', conn, params=(datos['dni'],))
        if movs.empty:
            return f"{datos['nombre']} no tiene movimientos para graficar."
        movs['fecha'] = pd.to_datetime(movs['fecha'])
        movs['monto_firmado'] = movs.apply(lambda r: r['monto'] if r['tipo'] == 'prestamo' else -r['monto'] if r['tipo'] == 'pago' else 0, axis=1)
        movs['saldo_estimado'] = movs['monto_firmado'].cumsum()
        plt.figure(figsize=(10, 4.8))
        ax = sns.lineplot(data=movs, x='fecha', y='saldo_estimado', marker='o')
        ax.set_title(f"Evolucion de deuda - {datos['nombre']}")
        ax.set_xlabel('Fecha')
        ax.set_ylabel('Saldo estimado (S/)')
        plt.xticks(rotation=25)
        plt.tight_layout()
        
        fecha_archivo = date.today().isoformat()
        carpeta_cliente = carpeta_graficos_cliente(datos['nombre'], datos['dni'])
        ruta = carpeta_cliente / f"grafico_historial_prestamos_{nombre_seguro_archivo(datos['nombre'])}_{datos['dni']}_{fecha_archivo}.png"
        plt.savefig(ruta, dpi=160, bbox_inches='tight')
        plt.close()
        return f"Grafico historico de {datos['nombre']} generado. Ruta: {ruta}"
    except Exception as e:
        return f"No se pudo generar grafico del cliente. Detalle: {e}"


@tool
def mostrar_rutas_dataset() -> str:
    """
    Muestra donde se guarda la base historica SQLite, los CSV exportados y la carpeta de graficos.
    Usala cuando el usuario pregunte donde revisar los datos.
    """
    exportar_dataset_csv()
    return (
        f"Base SQLite: {DB_PATH}\n"
        f"Clientes CSV: {CSV_CLIENTES}\n"
        f"Prestamos CSV: {CSV_PRESTAMOS}\n"
        f"Movimientos CSV: {CSV_MOVIMIENTOS}\n"
        f"Graficos: {GRAPH_DIR}"
    )



for _tool_direct in [
    consultar_deuda,
    consultar_historial_cliente,
    categorizar_cliente,
    listar_clientes_por_categoria,
    listar_principales_deudores,
    resumen_financiero,
    resumen_categorias_clientes,
    consultar_contacto_cobranza,
    mostrar_rutas_dataset,
    generar_grafico_morosidad,
    generar_grafico_cliente,
]:
    _tool_direct.return_direct = True


herramientas = [
    registrar_cliente,
    registrar_prestamo_fiado,
    registrar_pago,
    consultar_deuda,
    consultar_historial_cliente,
    categorizar_cliente,
    listar_principales_deudores,
    resumen_financiero,
    resumen_categorias_clientes,
    listar_clientes_por_categoria,
    consultar_contacto_cobranza,
    configurar_limite_credito,
    registrar_venta_contado,
    cerrar_caja,
    generar_grafico_morosidad,
    generar_grafico_cliente,
    mostrar_rutas_dataset,
]

len(herramientas)


## 5. Creacion del agente con memoria conversacional

El LLM interpreta el mensaje del bodeguero y decide que herramienta utilizar. Los calculos y cambios de datos se realizan en las herramientas, que leen y escriben en la base historica.


In [ ]:

from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


openrouter_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
if not openrouter_key:
    raise ValueError("Falta OPENROUTER_API_KEY. Ejecuta primero la celda 2 y pega tu clave sk-or-v1-...")

llm = ChatOpenAI(
    model="google/gemini-2.5-flash",
    openai_api_key=openrouter_key,
    base_url="https://openrouter.ai/api/v1",
    default_headers={"HTTP-Referer": "https://colab.research.google.com/", "X-Title": "Agente Fiados DTI"},
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un agente inteligente para una bodega peruana. Actuas como asistente administrativo del bodeguero. "
        "El sistema gestiona clientes, DNI, telefono, prestamos/fiados, pagos, deuda, categorias, estados, caja, dataset historico y graficos. "
        "Importante: el DNI debe tener exactamente 8 digitos y el telefono 9 digitos. "
        "Diferencia conceptos: monto_prestamo es lo que el cliente pide fiado; limite_credito es la politica del bodeguero. "
        "Cuando alguien quiere fiar o debe dinero usa registrar_prestamo_fiado. Cuando paga usa registrar_pago. "
        "La categoria de pago mide comportamiento historico: AD paga antes, A puntual, B retraso leve, C tardio, D riesgoso, NUEVO sin historial. "
        "El estado crediticio mide situacion actual: ACTIVO, OBSERVACION, CRITICO o BLOQUEADO. "
        "Si el usuario pide graficos, usa las herramientas de graficos; esas herramientas muestran la imagen directamente en la celda. "
        "Si el usuario pregunta donde revisar datos, usa mostrar_rutas_dataset. "
        "Cuando respondas resumenes, categorias, rankings o listas, usa lineas separadas con guiones. No lo escribas todo en una sola frase. "
        "Si falta cliente, DNI, monto o fecha, pregunta solo por el dato faltante. No inventes DNI, telefono ni montos. "
        "Si el mensaje es ambiguo como 'si', 'debe', 'pago' o 'saldo', pide una aclaracion concreta. "
        "Responde en espanol claro, breve y orientado a decision. Usa soles peruanos."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm=llm, tools=herramientas, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=herramientas,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=6,
)

chat_history = []
print("Agente creado correctamente.")


## 6. Seed de datos amplio para la demo

Esta seccion carga datos historicos de demostracion: clientes, DNI, telefonos, limites, prestamos, pagos, fechas de vencimiento, categorias y estados. El seed permite probar el agente sin registrar manualmente todos los casos antes de la exposicion.


In [ ]:
# Cambia a True si quieres que el notebook cargue automaticamente el dataset demo.
# Recomendado para exposicion: dejar True y ejecutar esta celda antes de probar el agente.
CARGAR_DATOS_PRUEBA = True

if CARGAR_DATOS_PRUEBA:
    print(cargar_seed_demo())
    print(mostrar_rutas_dataset.invoke({}))
else:
    print("Seed no cargado. Puedes ejecutar cargar_seed_demo() manualmente cuando quieras probar con datos historicos.")


## 7. Prueba directa del agente

Esta seccion permite ejecutar una consulta rapida para verificar que el agente reconoce frases naturales y selecciona herramientas correctamente.


In [ ]:
def mensaje_ambiguo(mensaje: str) -> str | None:
    texto = re.sub(r"\s+", " ", str(mensaje).strip().lower())
    texto = texto.strip("!.,;:")
    entradas_minimas = {"si", "si", "ok", "debe", "deuda", "saldo", "pago", "pago", "fiado", "consulta"}
    if texto in entradas_minimas:
        return (
            "Necesito un poco mas de informacion. Por ejemplo: "
            "'Juan me debe 45 soles por aceite', 'Carmen pago 30 soles' o 'Cuanto debe Juan'."
        )
    if re.fullmatch(r"cuanto debe", texto):
        return "De que cliente quieres consultar la deuda? Por ejemplo: 'Cuanto debe Juan'."
    return None


def limpiar_respuesta_repetida(texto: str) -> str:
    limpio = str(texto).strip()
    mitad = len(limpio) // 2
    if len(limpio) > 20 and len(limpio) % 2 == 0 and limpio[:mitad].strip() == limpio[mitad:].strip():
        return limpio[:mitad].strip()
    partes = limpio.split("\n")
    if len(partes) >= 2 and len(partes) % 2 == 0:
        primera = "\n".join(partes[: len(partes)//2]).strip()
        segunda = "\n".join(partes[len(partes)//2 :]).strip()
        if primera == segunda:
            return primera
    return limpio


def preguntar_agente(mensaje: str) -> str:
    global chat_history
    aclaracion = mensaje_ambiguo(mensaje)
    if aclaracion:
        chat_history.append(HumanMessage(content=mensaje))
        chat_history.append(AIMessage(content=aclaracion))
        return aclaracion

    respuesta = agent_executor.invoke({"input": mensaje, "chat_history": chat_history})
    salida = limpiar_respuesta_repetida(respuesta.get("output", ""))
    chat_history.append(HumanMessage(content=mensaje))
    chat_history.append(AIMessage(content=salida))
    return salida


print("Funcion preguntar_agente lista. Prueba sugerida: preguntar_agente('Juan me debe 45 soles por aceite')")


## 8. Bucle interactivo para la demo

Antes de conversar con el agente se solicita inicio de sesion del bodeguero. Solo un usuario autorizado puede registrar prestamos, pagos o consultar informacion de clientes.

Usuarios de demo:

- Usuario: `admin` / Clave: `1234`
- Usuario: `bodeguero` / Clave: `2026`

Para cerrar la sesion se debe escribir `salir`.


In [ ]:
from IPython.display import Image, display
import ipywidgets as widgets
from getpass import getpass as pedir_clave


def mostrar_imagen_si_existe(texto: str) -> bool:
    # Detecta rutas PNG aunque el agente diga "Ruta:", "Archivo guardado en:" o escriba la ruta directa.
    rutas = re.findall(r"([A-Za-z]:\\[^\n]+?\.png|/[^\n]+?\.png)", str(texto))
    imagen_mostrada = False

    for ruta_texto in rutas:
        ruta_texto = ruta_texto.strip().strip('`').strip()
        candidatos = [Path(ruta_texto)]

        # Si Colab conserva una ruta antigua en memoria, se busca el mismo archivo dentro de la carpeta de graficos.
        nombre_archivo = Path(ruta_texto).name
        if nombre_archivo:
            candidatos.extend(sorted(GRAPH_DIR.rglob(nombre_archivo), key=lambda p: p.stat().st_mtime, reverse=True))

        for ruta in candidatos:
            if ruta.exists():
                try:
                    display(Image(filename=str(ruta), width=650))
                    imagen_mostrada = True
                    break
                except Exception as e:
                    print(f"Agente: El grafico fue generado, pero no se pudo mostrar automaticamente. Detalle: {e}")

    if not imagen_mostrada and 'grafico' in str(texto).lower():
        recientes = sorted(GRAPH_DIR.rglob('*.png'), key=lambda p: p.stat().st_mtime, reverse=True)
        if recientes:
            try:
                display(Image(filename=str(recientes[0]), width=650))
                imagen_mostrada = True
            except Exception as e:
                print(f"Agente: El grafico fue generado, pero no se pudo mostrar automaticamente. Detalle: {e}")

    return imagen_mostrada


print("Sistema de seguridad del agente de prestamos y cobros")
print("Inicia sesion para continuar.\n")

intentos = 0
while not sesion_bodeguero.get('activa') and intentos < 3:
    usuario_login = input("Usuario bodeguero: ").strip()
    clave_login = pedir_clave("Clave: ").strip()
    mensaje_login = iniciar_sesion_bodeguero(usuario_login, clave_login)
    print(mensaje_login + "\n")
    intentos += 1

if not sesion_bodeguero.get('activa'):
    print("No se pudo iniciar sesion. Vuelve a ejecutar esta celda para intentarlo nuevamente.")
else:
    usuario_actual = sesion_bodeguero.get('usuario') or 'bodeguero'
    print("============================================================")
    print(f"Agente: Hola, {usuario_actual}. Soy el agente de prestamos y cobros.")
    print("============================================================\n")

    print("Servicios disponibles:")
    print("  - Registrar clientes, prestamos/fiados y pagos.")
    print("  - Consultar deudas, historial, categorias y contactos de cobranza.")
    print("  - Generar resumenes financieros y graficos.")
    print("  - Mostrar la ubicacion del dataset historico.\n")

    print("Consulta inicial:")
    print("  Que movimiento realizaremos hoy?\n")

    print("Cierre de sesion:")
    print("  Si desea cerrar sesion, escriba 'salir'.")
    print("============================================================\n")

    entrada_consulta = widgets.Text(
        value='',
        placeholder="Escribe aqui la consulta del bodeguero",
        description='Bodeguero:',
        layout=widgets.Layout(width='75%')
    )
    boton_enviar = widgets.Button(
        description='Enviar',
        button_style='primary',
        layout=widgets.Layout(width='110px')
    )
    salida_chat = widgets.Output(
        layout=widgets.Layout(
            border='1px solid #555',
            padding='10px',
            width='95%',
            max_height='520px',
            overflow_y='auto'
        )
    )

    display(widgets.HBox([entrada_consulta, boton_enviar]))
    display(salida_chat)

    def procesar_consulta(_=None):
        usuario = entrada_consulta.value.strip()
        entrada_consulta.value = ''
        if not usuario:
            return

        with salida_chat:
            print(f"Bodeguero: {usuario}")

            if usuario.lower() in ["salir", "exit", "terminar"]:
                print("Agente: Gracias por usar el asistente de prestamos y cobros. Sesion finalizada. Hasta luego, tenga buen dia.\n")
                entrada_consulta.disabled = True
                boton_enviar.disabled = True
                return

            try:
                respuesta = preguntar_agente(usuario)
                print(f"Agente:\n{respuesta}\n")
                if mostrar_imagen_si_existe(respuesta):
                    print("Agente: Grafico mostrado. Puede escribir otra consulta o 'salir'.\n")
            except Exception as e:
                print(f"Agente: Ocurrio un error controlado en la conversacion: {e}\n")

    boton_enviar.on_click(procesar_consulta)
    try:
        entrada_consulta.on_submit(procesar_consulta)
    except Exception:
        pass


## 9. Frases sugeridas para la sustentacion

Frases de prueba recomendadas despues de cargar el seed:

- `Dame un resumen financiero.`
- `Quienes son mis principales deudores?`
- `Consulta la deuda del DNI 99887766.`
- `Categoriza al cliente con DNI 11223344.`
- `Dame el contacto de cobranza del DNI 99887766.`
- `Genera un grafico de morosidad.`
- `Genera un grafico del cliente con DNI 12345678.`
- `Muestrame donde esta guardado el dataset.`
- `Registra un prestamo fiado de 40 soles para Juan Perez con DNI 12345678 por arroz.`
- `Carmen Ruiz con DNI 87654321 pago 30 soles.`
